In [2]:
import sys, os, subprocess, threading
sys.path.append('backend/app/')
import xarray as xr, numpy as np
from backend.app.services import functions
from backend.app.config import PROJECT_DES, DELFT_PATH

In [2]:
# Stream logs
def stream_logs(process, input_dir):
    try:
        for line in process.stdout:
            line = line.strip()
            if not line: continue
            print(line)
    except Exception as e:
        print(f"[INTERNAL ERROR] {e}")
    finally:
        process.wait()
        print("Reorganizing outputs. Please wait...")
        try:
            post_result = functions.postProcess(input_dir)
            print(post_result["message"])
        except Exception as e:
            print(f"Simulation failed: {e}")

In [ ]:
project = '3months'
dir = os.path.join(PROJECT_DES, f"backend/projects/vlnguyen/{project}")
input_dir = os.path.normpath(os.path.join(dir, "input"))
mdu_path = os.path.normpath(os.path.join(input_dir, "FlowFM.mdu"))
bat_path = os.path.normpath(os.path.join(DELFT_PATH, "x64/dflowfm/scripts/run_dflowfm.bat"))
# bat_path = os.path.normpath(os.path.join(DELFT_PATH, "install_fm-suite_2026_02/bin/run_dflowfm.bat"))
# bat_path = os.path.normpath(os.path.join(DELFT_PATH, "install_fm-suite_2025_01/bin/run_dflowfm.bat"))
command = ["cmd.exe", "/c", bat_path, "--autostartstop", mdu_path]

In [ ]:
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    encoding="utf-8", errors="replace", bufsize=1, cwd=input_dir)
threading.Thread(target=stream_logs, args=(process, input_dir), daemon=True).start()

In [2]:
path = r"C:\Users\vanln\Desktop\Grid.nc"
ds = xr.open_dataset(path)
ds

<xarray.Dataset> Size: 212kB
Dimensions:            (mesh2d_nEdges: 3806, Two: 2, mesh2d_nFaces: 2339,
                        mesh2d_nMax_face_nodes: 3, mesh2d_nNodes: 1469)
Coordinates:
    mesh2d_node_x      (mesh2d_nNodes) float32 6kB ...
    mesh2d_node_y      (mesh2d_nNodes) float32 6kB ...
Dimensions without coordinates: mesh2d_nEdges, Two, mesh2d_nFaces,
                                mesh2d_nMax_face_nodes, mesh2d_nNodes
Data variables:
    crs                int32 4B ...
    mesh2d_edge_x      (mesh2d_nEdges) float32 15kB ...
    mesh2d_edge_y      (mesh2d_nEdges) float32 15kB ...
    mesh2d_edge_nodes  (mesh2d_nEdges, Two) int32 30kB ...
    mesh2d_face_nodes  (mesh2d_nFaces, mesh2d_nMax_face_nodes) float32 28kB ...
    mesh2d_edge_faces  (mesh2d_nEdges, Two) int32 30kB ...
    mesh2d_face_x      (mesh2d_nFaces) float32 9kB ...
    mesh2d_face_y      (mesh2d_nFaces) float32 9kB ...
    mesh2d_face_x_bnd  (mesh2d_nFaces, mesh2d_nMax_face_nodes) float32 28kB ...
    mesh2d_face_y_bnd  (mesh2d_nFaces, mesh2d_nMax_face_nodes) float32 28kB ...
    mesh2d             int32 4B ...
    mesh2d_node_z      (mesh2d_nNodes) float32 6kB ...
Attributes:
    institution:  Private
    references:   vanlnNTNU@gmail.com

In [3]:
grid = functions.unstructuredGridCreator(ds)
grid
# grid.to_file('grid.geojson', driver='GeoJSON')

,geometry
0,"POLYGON ((6.48455 62.4831, 6.48595 62.48304, 6..."
1,"POLYGON ((6.48595 62.48304, 6.48585 62.48372, ..."
2,"POLYGON ((6.48455 62.4831, 6.48488 62.48376, 6..."
3,"POLYGON ((6.48595 62.48304, 6.48622 62.48368, ..."
4,"POLYGON ((6.48595 62.48304, 6.48687 62.48302, ..."
...,...
2334,"POLYGON ((6.45798 62.47496, 6.45993 62.47416, ..."
2335,"POLYGON ((6.48248 62.47968, 6.4862 62.47918, 6..."
2336,"POLYGON ((6.46222 62.47459, 6.46403 62.47485, ..."
2337,"POLYGON ((6.45993 62.47416, 6.46222 62.47459, ..."


In [3]:
path = r"backend\projects\vlnguyen\3months\input\Courser1_net.nc"
ds = xr.open_dataset(path)
ds

<xarray.Dataset> Size: 61kB
Dimensions:                      (nNetNode: 740, nNetLink: 2041,
                                  nNetLinkPts: 2, nNetElem: 1302,
                                  nNetElemMaxNode: 3, nNetCompLineNode_1: 177)
Dimensions without coordinates: nNetNode, nNetLink, nNetLinkPts, nNetElem,
                                nNetElemMaxNode, nNetCompLineNode_1
Data variables:
    Mesh2D                       int32 4B ...
    NetNode_x                    (nNetNode) float64 6kB ...
    NetNode_y                    (nNetNode) float64 6kB ...
    NetNode_z                    (nNetNode) float64 6kB ...
    NetLinkType                  (nNetLink) int32 8kB ...
    NetLink                      (nNetLink, nNetLinkPts) int32 16kB ...
    crs                          int32 4B ...
    NetElemNode                  (nNetElem, nNetElemMaxNode) int32 16kB ...
    NetCompLine_x_1              (nNetCompLineNode_1) float64 1kB ...
    NetCompLine_y_1              (nNetCompLineNode_1) float64 1kB ...
    projected_coordinate_system  int32 4B ...
Attributes:
    Spherical:    0
    Conventions:  UGRID-0.9

In [4]:
ds.close()